<a href="https://colab.research.google.com/github/mutiacyn/Uji-Validitas-dan-Reabilitas/blob/main/probabilitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pingouin factor_analyzer -q

In [2]:
!pip install --upgrade numpy scipy pingouin factor_analyzer

In [3]:
import pandas as pd, numpy as np, pingouin as pg
import re
from scipy import stats
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [4]:
from google.colab import files
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC).xlsx to Blank Quiz (Responses 3 TIC).xlsx
File terbaca: Blank Quiz (Responses 3 TIC).xlsx


In [49]:
df = pd.read_excel(nama_file)

JUMLAH_KOLOM_DEMOGRAFI = 5

# Mengambil kolom pernyataan setelah kolom demografi
butir_awal = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

# Memisahkan pertanyaan durasi gadget
kolom_durasi = [
    c for c in butir_awal.columns
    if "Durasi (jam)" in c
]

# Mengambil hanya butir Likert
butir = butir_awal.drop(columns=kolom_durasi)

print("Jumlah responden :", len(df))
print("Jumlah kolom Excel:", df.shape[1])
print("Jumlah butir Likert:", butir.shape[1])
print("Jumlah data kosong:", int(butir.isna().sum().sum()))
print("Kolom durasi yang dipisahkan:", len(kolom_durasi))

Jumlah responden : 40
Jumlah kolom Excel: 35
Jumlah butir Likert: 29
Jumlah data kosong: 0
Kolom durasi yang dipisahkan: 1


In [50]:
skala = {
    "Sangat Tidak Setuju (STJ)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
display(num.head())

Sel yang gagal dikonversi: 0


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Respon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Respon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Respon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Respon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Respon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5,5,4,4,4,4,4,5,4,4,...,2,5,4,5,4,5,5,4,4,5
1,4,4,4,4,4,4,4,5,4,4,...,4,2,4,4,4,5,4,5,5,5
2,5,5,4,4,4,4,4,4,4,4,...,2,4,5,4,2,5,5,5,4,5
3,5,4,5,2,4,5,5,4,4,2,...,2,4,4,4,4,5,4,4,4,4
4,5,5,4,4,5,5,5,4,2,4,...,2,2,2,2,4,5,5,4,5,5


In [51]:
dimensi = {}
for kolom in butir.columns:
    cocok = re.match(r"^(.*?) \[", kolom)
    nama = (cocok.group(1) if cocok else kolom).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Respon emosional: 5 butir


In [52]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n-2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])

        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
display(tabel_validitas)

n = 40 | r tabel (α=0,05; df=38) = 0.312


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.419,0.222,0.0071,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.390,0.224,0.0129,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.626,0.418,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.506,0.205,0.0009,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.793,0.597,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.634,0.424,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.231,-0.061,0.1510,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.578,0.437,0.0001,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.708,0.541,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.804,0.651,0.0000,Valid


In [53]:
def cronbach_alpha(x):
    k = x.shape[1]
    if k <= 1:
        return np.nan
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    if pd.isna(a):
        return "Tidak dapat diuji (Single-item)"
    return "Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80 else "Reliabel" if a >= 0.70 else "Tidak reliabel"

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    if len(daftar) < 2:
        # Lewati atau tandai dimensi dengan 1 butir saja agar tidak ZeroDivisionError
        hasil_reliabilitas.append({
            "Dimensi": nama,
            "Jumlah butir": len(daftar),
            "Cronbach's alpha": np.nan,
            "Kategori": "Tidak dapat diuji (Single-item)"
        })
        continue

    x = num[daftar]
    a = cronbach_alpha(x)

    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3) if not pd.isna(a) else np.nan,
        "Kategori": kategori(a)
    })

    if len(daftar) > 2:
        for i, kolom in enumerate(daftar, 1):
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(kolom, axis=1)), 3)
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik.,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.531
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.533
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.456
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.547
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.341
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.453
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.550


LATIHAN

NO 3

In [61]:

# Menghapus satu butir tidak valid: B7 pada dimensi Aspek Psikologis

nama_dimensi = "Penggunaan AI pada tugas atau projek kuliah"

# Mengambil daftar kolom pada dimensi
kolom_dimensi = dimensi[nama_dimensi]

# Menghitung Cronbach's Alpha sebelum penghapusan
data_awal = num[kolom_dimensi]
alpha_sebelum = cronbach_alpha(data_awal)

# Menghapus butir B7 (indeks ke-6)
kolom_baru = kolom_dimensi[:6]
data_setelah = num[kolom_baru]

# Menghitung Cronbach's Alpha setelah penghapusan
alpha_sesudah = cronbach_alpha(data_setelah)

# Menampilkan hasil
hasil_penghapusan = pd.DataFrame({
    "Keterangan": [
        "Sebelum penghapusan",
        "Setelah penghapusan B7"
    ],
    "Jumlah Butir": [
        len(kolom_dimensi),
        len(kolom_baru)
    ],
    "Cronbach's Alpha": [
        alpha_sebelum,
        alpha_sesudah
    ]
})

display(hasil_penghapusan)


,Keterangan,Jumlah Butir,Cronbach's Alpha
0,Sebelum penghapusan,7,0.550491
1,Setelah penghapusan B7,6,0.639350


In [62]:

# Uji validitas setelah penghapusan B7

hasil_validitas_baru = []

total_dimensi = data_setelah.sum(axis=1)

for i, kolom in enumerate(kolom_baru):
    r_hitung, p_value = stats.pearsonr(
        data_setelah[kolom],
        total_dimensi
    )

    hasil_validitas_baru.append({
        "Dimensi": nama_dimensi,
        "Butir": f"B{i+1}",
        "r_hitung": round(r_hitung, 3),
        "r_tabel": round(r_tabel, 3),
        "Keterangan": (
            "Valid" if r_hitung > r_tabel else "Tidak Valid"
        )
    })

tabel_validitas_baru = pd.DataFrame(hasil_validitas_baru)

display(tabel_validitas_baru)


,Dimensi,Butir,r_hitung,r_tabel,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.447,0.312,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.474,0.312,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.612,0.312,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.578,0.312,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.811,0.312,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.616,0.312,Valid


In [63]:

# Perbandingan reliabilitas sebelum dan sesudah penghapusan

print("HASIL PERBANDINGAN RELIABILITAS")
print("--------------------------------")
print(f"Alpha sebelum penghapusan : {alpha_sebelum:.3f}")
print(f"Alpha setelah penghapusan : {alpha_sesudah:.3f}")

if alpha_sesudah >= 0.70:
    print("Kesimpulan: Reliabilitas meningkat dan memenuhi batas 0,70.")
else:
    print("Kesimpulan: Reliabilitas meningkat, tetapi belum mencapai 0,70.")


HASIL PERBANDINGAN RELIABILITAS
--------------------------------
Alpha sebelum penghapusan : 0.550
Alpha setelah penghapusan : 0.639
Kesimpulan: Reliabilitas meningkat, tetapi belum mencapai 0,70.


NO 4

In [64]:
# Mengambil 20 responden secara acak
num_20 = num.sample(20, random_state=1)

# Menghitung r tabel untuk 20 responden
n_20 = len(num_20)
df_20 = n_20 - 2

t_20 = stats.t.ppf(0.975, df_20)
r_tabel_20 = t_20 / np.sqrt(df_20 + t_20**2)

print("Jumlah responden:", n_20)
print("Derajat bebas:", df_20)
print("r tabel:", round(r_tabel_20, 3))

# Menguji validitas seluruh butir pada subset
hasil_subset = []

for nama, daftar in dimensi.items():

    data_dimensi = num_20[daftar]
    total = data_dimensi.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):

        r_hitung, p_value = stats.pearsonr(
            data_dimensi[kolom],
            total
        )

        hasil_subset.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r_hitung, 3),
            "r tabel": round(r_tabel_20, 3),
            "Keterangan": (
                "Valid" if r_hitung > r_tabel_20
                else "Tidak valid"
            )
        })

tabel_subset = pd.DataFrame(hasil_subset)

print("\nHASIL UJI VALIDITAS 20 RESPONDEN")
display(tabel_subset)

# Menghitung jumlah butir valid dan tidak valid
jumlah_valid = (
    tabel_subset["Keterangan"] == "Valid"
).sum()

jumlah_tidak_valid = (
    tabel_subset["Keterangan"] == "Tidak valid"
).sum()

print("Jumlah butir valid:", jumlah_valid)
print("Jumlah butir tidak valid:", jumlah_tidak_valid)

Jumlah responden: 20
Derajat bebas: 18
r tabel: 0.444

HASIL UJI VALIDITAS 20 RESPONDEN


,Dimensi,Butir,r hitung,r tabel,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.369,0.444,Tidak valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.218,0.444,Tidak valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.520,0.444,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.688,0.444,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.784,0.444,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.638,0.444,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.369,0.444,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.555,0.444,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.761,0.444,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.765,0.444,Valid


Jumlah butir valid: 23
Jumlah butir tidak valid: 6


In [65]:

# LATIHAN 4: PERBANDINGAN UKURAN SAMPEL

valid_40 = (
    tabel_validitas["Keterangan"] == "Valid"
).sum()

tidak_valid_40 = (
    tabel_validitas["Keterangan"] == "Tidak valid"
).sum()

valid_20 = (
    tabel_subset["Keterangan"] == "Valid"
).sum()

tidak_valid_20 = (
    tabel_subset["Keterangan"] == "Tidak valid"
).sum()

perbandingan = pd.DataFrame({
    "Keterangan": [
        "Jumlah responden",
        "r tabel",
        "Butir valid",
        "Butir tidak valid",
        "Persentase valid"
    ],
    "40 Responden": [
        len(num),
        round(r_tabel, 3),
        valid_40,
        tidak_valid_40,
        f"{valid_40 / len(tabel_validitas) * 100:.2f}%"
    ],
    "20 Responden": [
        len(num_20),
        round(r_tabel_20, 3),
        valid_20,
        tidak_valid_20,
        f"{valid_20 / len(tabel_subset) * 100:.2f}%"
    ]
})

display(perbandingan)


,Keterangan,40 Responden,20 Responden
0,Jumlah responden,40,20
1,r tabel,0.312,0.444
2,Butir valid,27,23
3,Butir tidak valid,2,6
4,Persentase valid,93.10%,79.31%


In [55]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as writer:
    gabung.to_excel(writer, sheet_name="Validitas & Alpha Item", index=False)
    tabel_reliabilitas.to_excel(writer, sheet_name="Reliabilitas Dimensi", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>